# ?? Model Explorer: Testing NLLB-3.3B, mT5-large & Aya-101
This notebook allows you to train and evaluate alternative state-of-the-art multilingual models:
1. **`facebook/nllb-200-3.3B`** (Full 3.3B Parameter NLLB model with LoRA PEFT)
2. **`google/mt5-large`** (Google Multilingual T5 Large)
3. **`CohereForAI/aya-101`** (Cohere 101-Language Instruction-Tuned Model)

In [ ]:
import os, sys, re, json, subprocess
from pathlib import Path
import pandas as pd, numpy as np, torch

try: _cwd = Path.cwd()
except (FileNotFoundError, OSError): os.chdir("/home/jovyan"); _cwd = Path.cwd()
repo_name = "Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages"
if (_cwd / "src").exists(): BASE_DIR = _cwd
elif (_cwd.parent / "src").exists(): BASE_DIR = _cwd.parent
elif (_cwd / repo_name / "src").exists(): BASE_DIR = _cwd / repo_name
elif (Path("/home/jovyan") / repo_name / "src").exists(): BASE_DIR = Path("/home/jovyan") / repo_name
else: BASE_DIR = Path("/home/jovyan")
os.chdir(BASE_DIR)
for p in [str(BASE_DIR), str(BASE_DIR / "src")]:
    if p not in sys.path: sys.path.insert(0, p)

DATA_DIR        = BASE_DIR / "data" / "raw"
SUBMISSIONS_DIR = BASE_DIR / "submissions"
CHECKPOINTS_DIR = BASE_DIR / "models" / "checkpoints"
SRC_PATH        = BASE_DIR / "src" / "nllb_pipeline.py"

from model_explorer import load_exploration_model, SUPPORTED_MODELS
print(f"BASE_DIR: {BASE_DIR.resolve()}")
print(f"CUDA Available: {torch.cuda.is_available()}")


In [ ]:
# Launch NLLB-200-3.3B LoRA PEFT Fine-Tuning
import subprocess

cmd_3_3b = [
    sys.executable, str(SRC_PATH),
    "--model_name", "facebook/nllb-200-3.3B",
    "--use_peft",
    "--use_rag",
    "--use_dense_rag",
    "--min_similarity", "0.25",
    "--epochs", "3",
    "--batch_size", "4",  # Smaller batch size for 3.3B parameter model
    "--learning_rate", "2e-4",
    "--train_path", str(DATA_DIR / "Training set.csv"),
    "--val_path", str(DATA_DIR / "Validation set.csv"),
    "--test_path", str(DATA_DIR / "Test set.csv"),
    "--output_dir", str(CHECKPOINTS_DIR / "nllb-3.3b-lora-checkpoint"),
    "--submission_path", str(SUBMISSIONS_DIR / "submission_nllb_3.3b_lora.csv"),
]

print("?? Launching NLLB-200-3.3B LoRA Fine-Tuning & Evaluation...")
process = subprocess.Popen(cmd_3_3b, cwd=str(BASE_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
